<a href="https://colab.research.google.com/github/chickens110/Intro-to-Python-week_1/blob/main/code1Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-google-genai tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.8 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:

from langchain_google_genai import ChatGoogleGenerativeAI # Gemini LLM wrapper for LangChain
from langchain.agents import initialize_agent, Tool
from langchain.memory import ConversationBufferMemory
from tavily import TavilyClient

In [ ]:
import os
from getpass import getpass


GOOGLE_API_KEY = getpass("Enter your Google Gemini API key: ")
TAVILY_API_KEY = getpass("Enter your Tavily API key: ")


os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY


Enter your Google Gemini API key: ··········
Enter your Tavily API key: ··········


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.9) #Lower = more factual, higher = more creative


In [ ]:

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [ ]:
#  Tavily search as a LangChain Tool
# Agents use "tools" to extend their abilities
# For example: calculator, Wikipedia, Tavily search

def tavily_search_tool(query: str) -> str:
    """Search the web using Tavily."""
    result = tavily_client.search(query)
    return result['results'][0]['content'] if result['results'] else "No results found."

# def calculator_tool(query: str) -> str:
#     """Run a mathematical operation."""
#     return str(eval(query))

tools = [
    Tool(
        name="web_search",
        func=tavily_search_tool,
        description="Useful for searching real-time information on the internet."
    )
    # Tool(
    #     name="calculator",
    #     func=calculator_tool,
    #     description="Useful for when you need to answer questions about math."
    # )

]


In [ ]:
# Conversation Memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

#  Tools + Memory
agent = initialize_agent(
    tools,
    llm,
    agent="chat-conversational-react-description",
    memory=memory,
    verbose=True
)


In [ ]:
# question = "how can i prepare rice without water"
# answer = agent.run(question)
# print(answer)

q1 = "Who founded Tesla?"
resp1 = agent.run(q1)
print("Q1:", q1)
print("A1:", resp1)

q2 = "What are their most famous achievements?"
resp2 = agent.run(q2)
print("\nQ2:", q2)
print("A2:", resp2)

q3 = "Summarize in 3 bullet points."
resp3 = agent.run(q3)
print("\nQ3:", q3)
print("A3:", resp3)




> Entering new AgentExecutor chain...
```json
{
  "action": "Final Answer",
  "action_input": "Tesla was founded by Martin Eberhard and Marc Tarpenning.  Elon Musk joined later and became a significant investor and eventually CEO."
}
```

> Finished chain.
Q1: Who founded Tesla?
A1: Tesla was founded by Martin Eberhard and Marc Tarpenning.  Elon Musk joined later and became a significant investor and eventually CEO.


> Entering new AgentExecutor chain...
```json
{
  "action": "web_search",
  "action_input": "Martin Eberhard and Marc Tarpenning achievements"
}
```
Observation: Britannica Money # Martin Eberhard and Marc Tarpenning Tarpenning was raised in Sacramento, Calif., and earned a bachelor’s degree (1985) in computer science from the University of California, Berkeley. In 1997 Eberhard and Tarpenning cofounded NuvoMedia, an e-book venture that produced the Rocket eBook (1998). Eberhard served as CEO and Tarpenning led development until 2000, when NuvoMedia was sold to Gemstar–

In [ ]:

print(agent.run("What was I asking about earlier?"))




> Entering new AgentExecutor chain...
```json
{
  "action": "Final Answer",
  "action_input": "Earlier, you asked about who founded Tesla and what their most famous achievements were."
}
```

> Finished chain.
Earlier, you asked about who founded Tesla and what their most famous achievements were.


In [ ]:
# app.py
import os
import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, Tool
from langchain_community.tools.tavily_search import TavilySearchResults

# Load API keys securely
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# Tools
tavily_tool = TavilySearchResults(api_key=TAVILY_API_KEY, max_results=3)
tools = [Tool(name="web_search", func=tavily_tool.run, description="Search the web")]

# Memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,

    agent="conversational-react-description",
    verbose=True  # capture reasoning
)

# Chat function with reasoning toggle
def chat_fn(message, history, show_reasoning):
    import io, sys
    old_stdout = sys.stdout
    sys.stdout = mystdout = io.StringIO()

    try:
        result = agent.run(message)
    finally:
        sys.stdout = old_stdout

    reasoning_trace = mystdout.getvalue()
    if show_reasoning:
        result = f"**Answer:** {result}\n\n---\n🧠 **Agent Reasoning Trace:**\n```\n{reasoning_trace}\n```"

    return result

# Gradio UI
with gr.Blocks(theme="soft") as demo:
    gr.Markdown(
        """
        # 🤖 AI Agent Demo
        Chat with an agent that has **tools, memory, and reasoning**.
        Toggle reasoning to see how it thinks step-by-step!
        """
    )

    with gr.Row():
        chatbot = gr.Chatbot(label="Agent Chat", height=400)

    with gr.Row():
        msg = gr.Textbox(label="Your Question")
    with gr.Row():
        show_reasoning = gr.Checkbox(label="Show Agent Reasoning", value=False)
    with gr.Row():
        submit = gr.Button("Ask")

    # Wire events
    def respond(message, history, show_reasoning):
        response = chat_fn(message, history, show_reasoning)
        history = history + [(message, response)]
        return history, ""

    submit.click(respond, [msg, chatbot, show_reasoning], [chatbot, msg])

demo.launch(share=True)
